# KYC Presenter Walkthrough

Deliver this after the fraud-ring detection portion of the demo, once the audience has seen Louvain find ring 0. Three demos: identity is structure, identity explains the ring, and the knowledge layer explains the classification.

Run `06_kyc_load` first so the identity properties, knowledge layer, and gold columns exist. This notebook repeats the setup cells so it runs on its own.

## 0. Setup and Connect

In [ ]:
%pip install graphdatascience --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG   = "graph-on-databricks"
SCHEMA    = "graph-enriched-schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

NEO4J_OPTS = {
    "url":                    NEO4J_URI,
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
}

# Split the Neo4j read into concurrent tasks; label mode partitions internally
# by id range so the output is unchanged.
NEO4J_READ_PARTITIONS = "8"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

In [ ]:
from graphdatascience import GraphDataScience
from pyspark.sql import functions as F

gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print(f"Connected  |  GDS version: {gds.version()}")

---
## Demo 1: Sharing is structure, not a computed count

*"In the warehouse, a phone number is text in a column. To find who shares one, you self-join the table against itself, and every extra hop is another join. In the graph, the phone number is its own node. Everyone who uses it points at it. Sharing is something you see, not something you compute."*

```text
WAREHOUSE (self-join every row against every other row):
   368  312-555-0142 ┐
   927  312-555-0142 ├─ compare N×N to find matches
  1033  312-555-0142 ┘

GRAPH (the phone is its own node; sharing is visible):
    368 ─┐
    927 ─┤
         ├──► (312-555-0142)     4 arrows in  = SHARED
   1033 ─┤
   1696 ─┘

   5051 ───► (312-555-9999)      1 arrow in   = normal, unique
```

**ELI5 of the query:** Start at every customer and walk the one line to their phone node. Group those customers by the phone they landed on, so each phone carries the list of everyone pointing at it. Keep only the phones with more than one customer on the list. Those are the shared numbers, and the list is who shares them. No self-join and no comparing rows: the shared phone is already a single node with everyone attached to it.

Expected: two rows, one per planted phone. `312-555-0142` returns the four customers behind accounts 368, 927, 1033, 1696; `312-555-0143` the four behind 2184, 2216, 2612, 3003. No background phone appears, because background customers each hold a unique number.

In [ ]:
display(gds.run_cypher("""
    MATCH (c:Customer)-[:HAS_PHONE]->(p:Phone)
    WITH p, collect(DISTINCT c.name) AS customers
    WHERE size(customers) > 1
    RETURN p.number AS phone, customers
"""))

### Demo 2: WCC resolves the identity, and it explains the ring

*"GDS ran Weakly Connected Components over the identity graph. A normal customer is a household of one. These eight are one household, tied together by two phones and one shared address."*

**ELI5 of WCC:** Weakly Connected Components is the "who can reach whom" test. Start at any node, walk any relationship in either direction, and everyone you can reach belongs to your group. GDS does this for every node at once and stamps each one with a group id. A customer who shares no phone or address reaches no one, so they are a household of one. The eight story accounts can all reach each other through shared phones and a shared address, so GDS gives them the same id: one household.

```text
   368 ─┐                         ┌─ 2184
   927 ─┤                         ├─ 2216
      (Phone A)               (Phone B)
  1033 ─┤                         ├─ 2612
  1696 ─┘                         └─ 3003
      │                               │
      └────────► (Address X) ◄────────┘
                 bridges both halves → ONE household of 8

  normal:  5051 ─► (own phone)   (own address)     household of 1
```

No single phone touches all eight: Phone A binds four, Phone B binds another four, and the shared address is the bridge that fuses them. Remove the address and you would see two households of four.

**ELI5 of the query:** Look up the eight story accounts by id and read back the four labels GDS already wrote onto them. `cluster` is which household the account's owner landed in, `cluster_size` is how many customers are in that household, and the two shared counts say how many other customers reach this one through a shared phone or a shared address. Nothing is computed here; the query just reads the answers the graph algorithm stored.

Expected: all eight rows carry the same `cluster` id and `cluster_size` = 8. `shared_phones` = 3 on every row; `shared_addresses` = 3 on 1033, 1696, 2184, 2216 and 0 on the other four. One household, eight members, no single phone connects all eight; the address is what makes it one ring.

In [ ]:
STORY_ACCOUNTS = [368, 927, 1033, 1696, 2184, 2216, 2612, 3003]

display(gds.run_cypher("""
    MATCH (a:Account)
    WHERE a.account_id IN $ids
    RETURN a.account_id AS account,
           a.identity_cluster_id AS cluster,
           a.identity_cluster_size AS cluster_size,
           a.shared_phone_count AS shared_phones,
           a.shared_address_count AS shared_addresses
    ORDER BY a.account_id
""", params={"ids": STORY_ACCOUNTS}))

**The flagship query: fraud ring meets identity cluster.** Louvain already put these accounts in one transfer community. WCC now shows that eight of its accounts collapse into a single identity.

**ELI5 of the query:** Take every account the money-movement algorithm put in ring 0's community, then walk back to the customer who owns each one. Group those customers by their identity household and count how many customers sit in each household. Throw away the households with only one customer, the normal people. What is left is the household where several members of the same fraud community are secretly one identity, along with the account ids that give them away.

We read ring 0's Louvain `community_id` straight from the story accounts, since it changes on every re-projection, then ask: within that transfer community, which identity clusters hold more than one customer? Expect exactly one row, the story cluster of eight, because every other member of the community is its own identity cluster of one.

In [ ]:
# Ring 0's Louvain community = the community the story accounts sit in.
ring_row = gds.run_cypher("""
    MATCH (a:Account) WHERE a.account_id IN $ids
    RETURN a.community_id AS community, count(*) AS n
    ORDER BY n DESC LIMIT 1
""", params={"ids": STORY_ACCOUNTS})
ring_community = int(ring_row["community"].iloc[0])
print(f"Ring 0 Louvain community = {ring_community}")

display(gds.run_cypher("""
    MATCH (a:Account)
    WHERE a.community_id = $ring_community
    MATCH (c:Customer)-[:OWNS]->(a)
    WITH c.identity_cluster_id AS cluster,
         count(DISTINCT c) AS customers,
         collect(DISTINCT a.account_id) AS accounts
    WHERE customers > 1
    RETURN cluster, customers, accounts
""", params={"ring_community": ring_community}))

### Demo 3: The knowledge layer explains the classification

*"The customer's question is not only who violates the policy. It is which business definition and which data source made the call. That explanation is a traversal too."*

```text
(Customer 1033)
   │ CLASSIFIED_AS  "shares 3 phones + 3 address with 7 others"
   ▼
(BusinessTerm: Shared Identity Ring)
   ├─ DEFINED_BY ─► (BusinessRule: KYC-WCC-001)
   │                    │ DERIVED_FROM
   │                    ▼
   │               (DataSource: silver.customers.phone, .address)
   └─ GOVERNED_BY ─► (Policy: KYC-CIP-001, authority FinCEN 31 CFR 1020.220)
```

**ELI5 of the query:** Start at each flagged customer and follow the trail the classification left behind. First hop to the business term they were tagged with, then to the rule that defines that term, then out to the policy the term answers to and back to the source columns the rule reads. One walk returns the whole chain of custody: who was flagged, the plain-language reason, the term, the rule and its logic, the policy and its regulator, and the exact columns the decision rests on.

Expected: eight rows, one per violating customer. `why` is a plain-language reason like `shares 3 phone(s) and 3 address with 7 other customer(s) in identity cluster 42`. `business_term` is `Shared Identity Ring`, `rule` is `KYC-WCC-001`, `policy` is `KYC-CIP-001` under authority `FinCEN 31 CFR 1020.220`, and `data_sources` lists `silver.customers.phone` and `silver.customers.address`. No background customer appears.

In [ ]:
display(gds.run_cypher("""
    MATCH (c:Customer)-[cl:CLASSIFIED_AS]->(term:BusinessTerm)-[:DEFINED_BY]->(rule:BusinessRule)
    MATCH (term)-[:GOVERNED_BY]->(policy:Policy)
    MATCH (rule)-[:DERIVED_FROM]->(src:DataSource)
    RETURN c.customer_id      AS customer,
           cl.reason          AS why,
           term.name          AS business_term,
           rule.rule_id       AS rule,
           rule.logic         AS rule_logic,
           policy.policy_id   AS policy,
           policy.authority   AS policy_authority,
           collect(DISTINCT src.name) AS data_sources
    ORDER BY c.customer_id
"""))

---
## One-line Recap for the Room

Money movement detects the ring. Identity resolution proves it is one person. The knowledge layer names the policy and the source. Genie serves all three from Delta.